# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library. We'll step through the dataset's Croissant schema, inspect available record sets, and carry out basic analyses and visualizations.

### Dataset Source
The dataset source is provided via a Croissant schema URL and uses the unique `@id` fields to reference every record set, field, and column.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset (creates a Croissant Dataset object)
dataset = mlc.Dataset(croissant_url)

# Access and print the basic metadata for a quick overview
metadata = dataset.metadata
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review the available record sets, fields, and their unique `@id` references. This helps identify the structure and field names for subsequent data loading and operations.

We'll walk through available record sets in the schema, print each record set's `@id`, its name, and display fields and columns for one record set as an example.

In [ ]:
# List all record sets and their `@id`
record_sets = []
record_sets_structure = []
for rs in metadata.record_sets:
    # Print the record set's @id and name
    print(f"Record set @id: {rs['@id']}, name: {rs.get('name','')}")
    record_sets.append(rs['@id'])
    # Gather structure for details on fields
    fields = rs.get('field', []) or []
    cols = rs.get('column', []) or []
    if fields or cols:
        record_sets_structure.append({'@id': rs['@id'], 'fields': fields, 'columns': cols})
if record_sets:
    print(f"\nExample structure for the first record set: {record_sets[0]}")
    rs0_struct = next(x for x in record_sets_structure if x['@id'] == record_sets[0])
    print(f"Fields (@id): {[f['@id'] if isinstance(f,dict) else f for f in rs0_struct['fields']]}")
    print(f"Columns (@id): {[c['@id'] if isinstance(c,dict) else c for c in rs0_struct['columns']]}")
else:
    print("No record sets found in the Croissant schema.")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. All datasets, fields, and columns should be referenced by their unique `@id`, as per the Croissant schema.

**Note:** Replace `<record_set_id>` with the actual `@id` from the overview.

In [ ]:
# In practice, you should copy @id(s) from the available record sets above. For demonstration:
if record_sets:
    dataframes = {}
    for record_set_id in record_sets:
        print(f"Loading records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
        else:
            print(f"No data found for record set: {record_set_id}")
    # Inspect columns for the first available DataFrame
    first_available = next(((k,v) for k,v in dataframes.items() if not v.empty), None)
    if first_available is not None:
        key, df = first_available
        print(f"First non-empty DataFrame columns (@id): {df.columns.tolist()}")
        display(df.head())
    else:
        print("No records were loaded into DataFrames.")
else:
    print("No record sets available to load.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps on the DataFrame. We'll filter for values in a numeric field and demonstrate normalization and grouping, referencing all fields by their `@id`.

**Important**: Replace `<numeric_field_id>` and `<group_field_id>` below with the relevant column names from your DataFrame (based on earlier cell output and the Croissant schema).

In [ ]:
import numpy as np

# Example selection of a numeric field (using the first non-empty DataFrame and picking a numeric column)
if first_available is not None:
    key, df = first_available
    # Try to guess a numeric column
    numeric_fields = df.select_dtypes(include=[np.number]).columns
    if len(numeric_fields) == 0:
        # If no numeric columns, print available columns for manual inspection
        print("No numeric fields found. Available columns:", df.columns.tolist())
        numeric_field_id = None
    else:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")

    # Apply a filter
    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered rows where {numeric_field_id} > average ({threshold:.2f}):")
        display(filtered_df.head())

        # Normalize this numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} (z-score):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a non-numeric field
        group_fields = df.select_dtypes(exclude=[np.number]).columns
        if len(group_fields) > 0:
            group_field_id = group_fields[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            display(grouped_df.head())
        else:
            print("No non-numeric fields to group by.")
    else:
        print("No numeric field to analyze.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships. We'll create a histogram and a scatter plot example if numeric columns are present.

**Note**: Replace field references as needed based on your outputs above and the Croissant schema.

In [ ]:
import matplotlib.pyplot as plt

if first_available is not None and numeric_field_id is not None:
    # Histogram
    plt.figure(figsize=(8, 4))
    plt.hist(df[numeric_field_id].dropna(), bins=25, color='skyblue', edgecolor='black')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If there is a second numeric field, create a scatter plot
    if len(numeric_fields) > 1:
        plt.figure(figsize=(6, 6))
        plt.scatter(df[numeric_fields[0]], df[numeric_fields[1]], alpha=0.5)
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.title(f"{numeric_fields[0]} vs {numeric_fields[1]}")
        plt.show()
else:
    print("Insufficient data for visualization. Please check previous steps.")

## 6. Conclusion
In this notebook, we've demonstrated how to load, explore, and analyze the FAIR² dataset using `mlcroissant` and Python. 

- We referenced all structural elements by their `@id`s for transparency and reproducibility.
- Example EDA and visualization steps were provided for immediate analysis.

The approach shown here can be used for any Croissant-compliant dataset, by adjusting record set and field `@id`s as given in your dataset's schema structure.

For more advanced analyses, consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/python/) and adapt field selection and processing to your research needs.